# Chapter 11 · How big a step?
**Deck 2 · Live demo.** Race four step sizes and find where learning breaks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})

## 1 · Your prediction
Which is the smallest of these step sizes that **explodes**: 0.001, 0.01, 0.04, 0.055?

In [ ]:
my_prediction = None

In [ ]:
hours = np.array([2., 4., 6.]); marks = np.array([50., 60., 70.])
loss = lambda w, b: np.mean((w * hours + b - marks) ** 2)
def slopes(w, b):
    e = w * hours + b - marks
    return 2 * np.mean(e * hours), 2 * np.mean(e)

def run(eta, steps):
    w = b = 0.0; losses = [loss(w, b)]
    for _ in range(steps):
        gw, gb = slopes(w, b); w, b = w - eta * gw, b - eta * gb
        losses.append(loss(w, b))
        if not np.isfinite(losses[-1]) or losses[-1] > 1e12:
            break
    return np.array(losses)

## 2 · The race

In [ ]:
for eta in [0.001, 0.01, 0.04, 0.055]:
    L = run(eta, 500)
    plt.semilogy(L, label=f'eta {eta}')
plt.ylim(1e-4, 1e8); plt.xlabel('step'); plt.ylabel('loss (log)'); plt.legend(); plt.show()

## 3 · Steps to reach Asha ≈ 80

In [ ]:
def steps_to_asha(eta, limit=100_000):
    w = b = 0.0
    for i in range(1, limit):
        gw, gb = slopes(w, b); w, b = w - eta * gw, b - eta * gb
        if not np.isfinite(w) or abs(w) > 1e9:
            return 'explodes'
        if abs(8 * w + b - 80) < 0.5 and abs(w - 5) < 0.1 and abs(b - 40) < 0.5:
            return i
table = {eta: steps_to_asha(eta) for eta in [0.001, 0.01, 0.04, 0.055]}
for eta, s in table.items():
    print(f'eta {eta:<6} → {s}')

## 4 · Find the cliff
Bisection: keep halving the gap between a step size that works and one that explodes.

In [ ]:
def explodes(eta):
    return isinstance(steps_to_asha(eta, 20_000), str)

lo, hi = 0.04, 0.055
# TODO: repeat 20 times: try the middle; if it explodes, it becomes the new hi, else the new lo
cliff = None

## 5 · PROOF

In [ ]:
assert table == {0.001: 16057, 0.01: 1604, 0.04: 400, 0.055: 'explodes'}, 'the slide table'
assert abs(cliff - 0.0512) < 0.0005, 'the safe limit is about 0.051'
assert my_prediction == 0.055
print('✓ proof: the race table and the cliff match the slides')

## 6 · Your turn
Try 0.05 and 0.052. Which side of the cliff is each?

In [ ]:
# TODO

## Recap
Too small crawls (~16,000 steps), 0.04 is fast (~400), and just past ~0.051 every step overshoots and the loss explodes.